# **CSCE 5218 / CSCE 4930 Deep Learning**

# **The Perceptron** (20 pt)


### Build the Perceptron Model

You will need to complete some of the function definitions below.  DO NOT import any other libraries to complete this.

In [14]:
import math
import itertools
import re


# Corpus reader, all columns but the last one are coordinates;
#   the last column is the label
def read_data(file_name):
    f = open(file_name, 'r')

    data = []
    # Discard header line
    f.readline()
    for instance in f.readlines():
        if not re.search('\t', instance): continue
        instance = list(map(int, instance.strip().split('\t')))
        # Add a dummy input so that w0 becomes the bias
        instance = [-1] + instance
        data += [instance]
    return data


# If the length of the two arrays are not the same, raise the error flag.
# Use zip function to pair array items by index
def dot_product(array1, array2):
    if len(array1) != len(array2):
        print(array1)
        print(array2)
        raise ValueError("The two arrays must be the same length!")
    return sum(a * b for a, b in zip(array1, array2))

# sigmoid(x) = 1/ (1+ exp(-x))
def sigmoid(x):
    return 1 / (1 + math.exp(-x))

# The output of the model, which for the perceptron is
# the sigmoid function applied to the dot product of
# the instance and the weights
def output(weight, instance):
    dot_sum = dot_product(weight, instance)
    return sigmoid(dot_sum)

# Predict the label of an instance; this is the definition of the perceptron
# you should output 1 if the output is >= 0.5 else output 0
def predict(weights, instance):
    act_out = output(weights, instance[0:len(weights)])
    return 1 if act_out >= 0.5 else 0

# Accuracy = percent of correct predictions
def get_accuracy(weights, instances):
    # You do not need to write code like this, but get used to it
    #print("Test instances", instances)
    correct = sum([1 if predict(weights, instance) == instance[-1] else 0
                   for instance in instances])
    return correct * 100 / len(instances)


# Train a perceptron with instances and hyperparameters:
#       lr (learning rate)
#       epochs
# The implementation comes from the definition of the perceptron
#
# Training consists on fitting the parameters which are the weights
# that's the only thing training is responsible to fit
# (recall that w0 is the bias, and w1..wn are the weights for each coordinate)
#
# Hyperparameters (lr and epochs) are given to the training algorithm
# We are updating weights in the opposite direction of the gradient of the error,
# so with a "decent" lr we are guaranteed to reduce the error after each iteration.
def train_perceptron(instances, lr, epochs):
    #Initialize all the weights to 0
    #Note: the size of the weights = # of features, listed by instances[0]
    # excluding the last one, which is the truth value
    # the -1 in the first index of each instance is the input for the bias
    len_weights= len(instances[0])-1
    weights = [0] * len_weights
    #print("Initial weights", weights)

    for _ in range(epochs):
        for instance in instances:
            #Calculate the predicted output based on with current weights
            # Then calculate the error of predicted vs. true value
            #print("instance:", instance)
            in_value = dot_product(weights, instance[0:len_weights])
            output = sigmoid(in_value)
            error = instance[-1] - output
            # For a single perceptron,
            # calculate the weight adjustment for each weight, based on:
            #  A. sigmoid function derivative - output * (1-output)
            #  B. learning rate - lr
            #  C. error between truth value and prediction - error
            #  D. input influence from feature i - instance[i]
            for i in range(0, len_weights):
                weights[i] += lr * error * output * (1-output) * instance[i]

    return weights

## Run it

In [15]:

#instances_tr = read_data("train.dat")
#instances_te = read_data("test.dat")

# Update the way to read data since I am running PyCharm on my windows PC.
# train.txt and test.txt are downloaded to local directory then use function read_data to load.
train_path = r"C:\Users\Mini-Berlink\Documents\UNT\26-Spring-CSCE5218\Assignment\HW3\train.txt"
test_path  = r"C:\Users\Mini-Berlink\Documents\UNT\26-Spring-CSCE5218\Assignment\HW3\test.txt"

instances_tr = read_data(train_path)
instances_te = read_data(test_path)

lr = 0.005
epochs = 200
weights = train_perceptron(instances_tr, lr, epochs)
accuracy = get_accuracy(weights, instances_te)
print(f"#tr: {len(instances_tr):3}, epochs: {epochs:3}, learning rate: {lr:.3f}; "
      f"Accuracy (test, {len(instances_te)} instances): {accuracy:.1f}")

#tr: 400, epochs: 200, learning rate: 0.005; Accuracy (test, 100 instances): 80.0


## Questions

Answer the following questions. Include your implementation and the output for each question.



### Question 1

In `train_perceptron(instances, lr, epochs)`, we have the follosing code:
```
in_value = dot_product(weights, instance)
output = sigmoid(in_value)
error = instance[-1] - output
```

Why don't we have the following code snippet instead?
```
output = predict(weights, instance)
error = instance[-1] - output
```

#### TODO Add your answer here (text only)
```
[Answer]Prediction has one more step that sets output to be 1 or 0.
It is based on sigmoid output vs. threashold (0.5 in our case).
In this cae, the "out(1-output)" is always 0 when using the predict results.
Therefore, the weight update will be consistently 0. The training won't work.
In summary, to calculate gradient descent & weight update, we needs the real activation function output, not predicted output.
```



### Question 2
Train the perceptron with the following hyperparameters and calculate the accuracy with the test dataset.

```
tr_percent = [5, 10, 25, 50, 75, 100] # percent of the training dataset to train with
num_epochs = [5, 10, 20, 50, 100]              # number of epochs
lr = [0.005, 0.01, 0.05]              # learning rate
```

TODO: Write your code below and include the output at the end of each training loop (NOT AFTER EACH EPOCH)
of your code.The output should look like the following:
```
# tr:  20, epochs:   5, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  20, epochs:  10, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  20, epochs:  20, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
[and so on for all the combinations]
```
You will get different results with different hyperparameters.

#### TODO Add your answer here (code and output in the format above)


In [16]:
# Code ANSWER. Output can be seen after running the code in this box
tr_percents = [5, 10, 25, 50, 75, 100] # percent of the training dataset to train with
num_epochs = [5, 10, 20, 50, 100]     # number of epochs
lrs = [0.005, 0.01, 0.05]  # learning rates

train_path = r"C:\Users\Mini-Berlink\Documents\UNT\26-Spring-CSCE5218\Assignment\HW3\train.txt"
test_path  = r"C:\Users\Mini-Berlink\Documents\UNT\26-Spring-CSCE5218\Assignment\HW3\test.txt"

instances_tr = read_data(train_path)
total_tr_size = len(instances_tr)

instances_te = read_data(test_path)
total_test_size = len(instances_te)

for lr in lrs:
  for tr_pct in tr_percents:
    tr_size = round(total_tr_size*tr_pct/100)
    tr_instances = instances_tr[0:tr_size]
    for epochs in num_epochs:
      # Training weights
      weights = train_perceptron(tr_instances, lr, epochs)
      # Calculate Accuracy
      accuracy = get_accuracy(weights, instances_te)
      print(f"#tr: {tr_size:3}, epochs= {epochs:3}, learning rate: {lr:.3f}; Accuracy (test, {total_test_size} instances): {accuracy:.1f}")


#tr:  20, epochs=   5, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs=  10, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs=  20, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs=  50, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  20, epochs= 100, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  40, epochs=   5, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  40, epochs=  10, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  40, epochs=  20, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  40, epochs=  50, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr:  40, epochs= 100, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr: 100, epochs=   5, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr: 100, epochs=  10, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
#tr: 100, epochs

### Question 3
Write a couple paragraphs interpreting the results with all the combinations of hyperparameters. Drawing a plot will probably help you make a point. In particular, answer the following:
- A. Do you need to train with all the training dataset to get the highest accuracy with the test dataset?
```
[Answer] No.
The highest accuracy we acheived is 80% when running with the test.txt and train.txt.
Based on the hyper-parameter scan, one case can achieve 80% accuracy without running all the training dataset, shown below:
```
#tr: 300, epochs= 100, learning rate: 0.010; Accuracy (test, 100 instances): 80.0


- B. How do you justify that training the second run obtains worse accuracy than the first one (despite the second one uses more training data)?
```
#tr: 100, epochs:  20, learning rate: 0.050; Accuracy (test, 100 instances): 71.0
#tr: 200, epochs:  20, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
```
```
[Answer]
In general, accuracy improves over more training data.
However, in real situations, accuracy improvement also relies on learning rate and number of epochs.
In this specific case, there is 10x difference in learning rate between the two runs: 0.05 vs 0.005.
It is no surprise that the case with 0.05 lr converges faster, especially when only 20 epochs were done.
In our hyper-parameter scan running only up to 100 epochs, lr=0.05 turns out to be the better one for this model and dataset.
If we extends the epochs to more than 100, the run with 0.005 lr will gradually catch up.
In my original code, I did run with full training dataset and 200 epochs with lr=0.005, it reaches 80% accuracy as well.
Here are the runs showing 100 epochs improves the accuracy with 200 training size,
 and the run with full traing dataset reaches 80% accuracy with 200 epochs even though lr=0.005.
```
#tr: 200, epochs= 100, learning rate: 0.005; Accuracy (test, 100 instances): 74.0
#tr: 400, epochs: 200, learning rate: 0.005; Accuracy (test, 100 instances): 80.0


- C. Can you get higher accuracy with additional hyperparameters (higher than `80.0`)?
```
[Answer] I could not achieve accuracy better than 80%, even after extending epochs to 400 with full training dataset.
```
- D. Is it always worth training for more epochs (while keeping all other hyperparameters fixed)?
```
[Answer] It is NOT always worthwhile.
More epochs will help, when training loss is decreasing and gradients are  meaningful, or not minimal.
When the model has learned the underlying structure of the data, gradients become minimal and more epochs will not help.
At this point, more epochs could mean overfitting and hurt accuracy against validation and testing data.

```


